# 01 — Setup Google Drive, Ekstraksi EXIF Foto HP, & Download Dataset
**Runtime yang disarankan:** CPU saja (agar tidak membuang kuota GPU Colab Free).

Notebook ini mengerjakan:
1. Mount Google Drive untuk menyimpan dataset, checkpoint, dan log secara permanen.
2. Ekstraksi **EXIF / metadata** dari folder 40+ foto HP Anda ke tabel CSV (`exif_metadata.csv`) sebagai bukti keaslian test set.
3. Download dataset utama (data latih + test set asli) dan **test set buatan Anda** dari Roboflow dalam 2 format sekaligus:
   - Format **YOLOv8** (untuk Model 1: YOLOv8-seg)
   - Format **COCO Segmentation** (untuk Model 2: Mask R-CNN & Model 3: Mask2Former)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/CPMK3_Instance_Segmentation'
for sub in ['datasets', 'raw_hp_photos', 'runs', 'logs', 'exports']:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)
print('Folder proyek siap:', PROJECT_DIR)


## Bagian A: Ekstraksi EXIF / Metadata Foto HP (Test Set Sendiri)
Upload minimal **40 foto asli dari kamera HP Anda** (jangan lewat WhatsApp karena WA menghapus EXIF; upload langsung dari HP ke Google Drive) ke folder:
`/content/drive/MyDrive/CPMK3_Instance_Segmentation/raw_hp_photos/`

*(Opsional: pisahkan dalam subfolder `normal/` dan `sulit/` agar otomatis tercatat kategorinya).*


In [ ]:
!pip install -q Pillow pandas

import glob
import pandas as pd
from PIL import Image, ExifTags

PHOTO_DIR = os.path.join(PROJECT_DIR, 'raw_hp_photos')
img_exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
photo_paths = []
for ext in img_exts:
    photo_paths.extend(glob.glob(os.path.join(PHOTO_DIR, '**', ext), recursive=True))

print(f'Total foto ditemukan: {len(photo_paths)}')
if len(photo_paths) < 40:
    print('PERINGATAN: Jumlah foto masih kurang dari 40 (syarat minimal 40 foto, min. 10 kondisi sulit).')

TAG_MAP = {v: k for k, v in ExifTags.TAGS.items()}
TARGET_TAGS = [
    'Make', 'Model', 'DateTimeOriginal', 'ExposureTime',
    'FNumber', 'ISOSpeedRatings', 'FocalLength', 'Orientation'
]

rows = []
for p in sorted(photo_paths):
    rel_path = os.path.relpath(p, PHOTO_DIR)
    folder = os.path.dirname(rel_path) or 'root'
    try:
        with Image.open(p) as img:
            w, h = img.size
            exif_raw = img._getexif() or {}
            row = {
                'filename': os.path.basename(p),
                'subfolder': folder,
                'width': w,
                'height': h,
                'format': img.format,
            }
            for tag_name in TARGET_TAGS:
                tag_id = TAG_MAP.get(tag_name)
                val = exif_raw.get(tag_id, None)
                row[tag_name] = str(val) if val is not None else ''
            rows.append(row)
    except Exception as e:
        rows.append({'filename': os.path.basename(p), 'subfolder': folder, 'error': str(e)})

df_exif = pd.DataFrame(rows)
exif_csv = os.path.join(PROJECT_DIR, 'logs', 'exif_metadata.csv')
df_exif.to_csv(exif_csv, index=False)
print('EXIF metadata disimpan ke:', exif_csv)
df_exif.head(10)


## Bagian B: Download Dataset dari Roboflow
Isi API Key, Workspace, Project, dan Version Roboflow Anda di bawah ini.
- **Dataset Utama**: dataset latih + validasi + test set asli (misal dataset baju/jaket/kaos atau Panoptic Indoor Segmentation v5).
- **Dataset Test HP**: proyek Roboflow berisi 40+ foto HP Anda yang sudah Anda anotasi dengan **nama kelas yang sama persis**.


In [ ]:
!pip install -q roboflow

from roboflow import Roboflow

# === ISI KONFIGURASI ROBOFLOW ANDA ===
ROBOFLOW_API_KEY = "ISI_API_KEY_ANDA"

# 1. Dataset Utama (Train / Valid / Test Asli)
MAIN_WORKSPACE = "ISI_WORKSPACE_UTAMA"
MAIN_PROJECT   = "ISI_PROJECT_UTAMA"
MAIN_VERSION   = 1

# 2. Dataset Test Set HP Anda (hasil anotasi sendiri di Roboflow)
CUSTOM_WORKSPACE = "ISI_WORKSPACE_TEST_HP"
CUSTOM_PROJECT   = "ISI_PROJECT_TEST_HP"
CUSTOM_VERSION   = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Download Dataset Utama (format YOLOv8 & COCO)
main_ver = rf.workspace(MAIN_WORKSPACE).project(MAIN_PROJECT).version(MAIN_VERSION)
main_yolo_dir = os.path.join(PROJECT_DIR, 'datasets', 'main_yolo')
main_coco_dir = os.path.join(PROJECT_DIR, 'datasets', 'main_coco')
main_ver.download("yolov8", location=main_yolo_dir, overwrite=True)
main_ver.download("coco-segmentation", location=main_coco_dir, overwrite=True)

# Download Test Set HP (format YOLOv8 & COCO)
custom_ver = rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT).version(CUSTOM_VERSION)
custom_yolo_dir = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo')
custom_coco_dir = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco')
custom_ver.download("yolov8", location=custom_yolo_dir, overwrite=True)
custom_ver.download("coco-segmentation", location=custom_coco_dir, overwrite=True)

print("Semua dataset berhasil diunduh ke Google Drive!")


## Bagian C: Verifikasi Kesesuaian Kelas (Main vs Test HP)
Pastikan urutan & nama kelas antara dataset utama dan test set HP **100% sama** agar evaluasi tidak keliru.


In [ ]:
import yaml, json

with open(os.path.join(PROJECT_DIR, 'datasets', 'main_yolo', 'data.yaml')) as f:
    main_yaml = yaml.safe_load(f)
with open(os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_yolo', 'data.yaml')) as f:
    custom_yaml = yaml.safe_load(f)

print("Kelas Dataset Utama :", main_yaml.get('names'))
print("Kelas Test Set HP   :", custom_yaml.get('names'))
assert main_yaml.get('names') == custom_yaml.get('names'), (
    "PERINGATAN: Nama/urutan kelas tidak sama! Perbaiki kelas di Roboflow sebelum training."
)
print("OK! Daftar kelas cocok 100%.")
